# Logistic Regression on Glass Dataset

In this lab we build a logistic regression classifier from scratch using numpy. The Glass Identification dataset has 9 chemical features and we convert it into a binary classification task.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Read the glass dataset from CSV
glass_df = pd.read_csv("glass.csv")

# Basic info about the dataset
print(glass_df.shape)
print(glass_df.columns)
glass_df.head()

(214, 10)
Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')


        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1

## Binary Label Creation

The dataset has 6 glass types (1–7). We simplify this to a binary problem: Type 1 glass is labeled 1, everything else is labeled 0.

In [2]:
# Label = 1 if Type 1, else 0
glass_df["label"] = (glass_df["Type"] == 1).astype(int)

# Drop the original Type column
glass_df = glass_df.drop(columns=["Type"])

# Split into features and target
features = glass_df.drop(columns=["label"]).values
target = glass_df["label"].values

## Train-Test Split and Feature Scaling

We use an 80/20 split. Feature scaling is important here — the 9 chemical properties have very different ranges, and without normalization the gradient updates would be unstable.

In [3]:
# 80% training, 20% testing
feat_train, feat_test, lbl_train, lbl_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

# Normalize using standard scaler
scaler = StandardScaler()
feat_train = scaler.fit_transform(feat_train)
feat_test  = scaler.transform(feat_test)

## Sigmoid Activation

The sigmoid function squashes any real value into the range (0, 1), which we interpret as the probability of belonging to class 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [4]:
def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

def compute_prob(X, weights, bias):
    # Linear combination followed by sigmoid
    z_val = X @ weights + bias
    prob  = sigmoid_fn(z_val)
    return prob

## Binary Cross-Entropy Loss and Gradient Update

Binary cross-entropy measures how far the predicted probabilities are from the true labels. The gradient of this loss w.r.t weights is simply $(p - y)$, which we use to update parameters.

In [5]:
def bce_loss(labels, probs):
    # Binary cross entropy
    return -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))

def gradient_step(X, labels, weights, bias, lr):
    probs = compute_prob(X, weights, bias)
    diff  = probs - labels

    # Gradient descent update
    weights = weights - lr * (X.T @ diff) / len(labels)
    bias    = bias    - lr * np.mean(diff)
    return weights, bias

## Training Loop

Weights and bias start at zero. Each epoch runs one full gradient update over the training set.

In [6]:
# Initialise parameters
weights = np.zeros(feat_train.shape[1])
bias    = 0.0
lr      = 0.1
n_epochs = 100

# Run gradient descent for n_epochs
for epoch in range(n_epochs):
    weights, bias = gradient_step(feat_train, lbl_train, weights, bias, lr)

## Threshold-Based Decision

We convert probabilities to class labels using a threshold. A stricter threshold (0.7) means the model needs higher confidence before predicting class 1 — useful when false positives are costly.

In [7]:
def apply_threshold(probs, thresh=0.5):
    return (probs >= thresh).astype(int)

# Predict probabilities on test set
test_probs = compute_prob(feat_test, weights, bias)

# Apply two different thresholds
preds_05 = apply_threshold(test_probs, 0.5)
preds_07 = apply_threshold(test_probs, 0.7)

print("Decisions at 0.5:", preds_05)
print("Decisions at 0.7:", preds_07)

Decisions at 0.5: [0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0
 1 0 0 0 0 1]
Decisions at 0.7: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0]
